In [ ]:
include("model_2d.jl")
using CSV
using DataFrames


In [ ]:

simulation_type_tup = (

    # simulation steps:

    physics = true,
    brownian = true,
    transcr = true,
    cell_division = true,
    affinity = false,
    anim = false,
    save_data = true,
    track_progress = false,

    # specification of which functions to use:

    # physics: 
        # cell devision, can be random_angle_cell_devision,  non_random_cell_devision
    cell_div_func = random_angle_cell_devision,
    func_orth = false,
    tf_split = true,
    
    # transcription:
        #signalling: can be nearest_neighbour (signalling_nn), distance_based (signalling_distance)
    sign_func = signalling_distance,
    check_distance = true,
    topography = false,

    # animation:
        #scatter plot is default
    draw_edges = true,
    draw_radius = false,

    # simulate to specific number of cells?
    cell_number_check = true,
    donator = true

)
### Paramater ###
#par = CSV.File("standard_parameter.csv") |> Dict
par = CSV.File("new_parameter.csv") |> Dict
par["max_cells"] = 300
par["q"] = 0.9
par["k"] = 0.1
par["save_intervall"] = 100
### Time ###
time_ss = Time_struct(1000,100)

### Initial conditions 2d ###
"""
cells_dict = Dict(
    "xy" => ([-0.1 0; 0.1 0; 0.05 0.05]),
    "r" => [0.4, 0.5, 0.7],
    "u" => [0.1, 0.1,0.1],
    "v" => [0.1, 0.1,0.1],
)
"""
# init cells for gamma = 1

cells_dict = Dict(
    "xy" => ([-0.1 0; 0.1 0; 0.05 0.05]),
    "r" => [0.4, 0.5, 0.7],
    "u" => [1.0, 1.0,1.0],
    "v" => [1.0, 1.0,1.0],
)

###  Inititial conditions 3d ###
"""
cells_dict = Dict(
    "xy" => ([-0.1 0 0.3; 0.1 0 0.3; 0.05 0.05 0.3; 0.2 0.2 0.2; -0.2 -0.2 -0.2]),
    "r" => [0.6, 0.6, 0.6, 0.6, 0.6],
    "u" => [0.1, 0.1,0.1, 0.1, 0.1],
    "v" => [0.1, 0.1,0.1, 0.1, 0.1],
)
"""
# init cells for gamma = 1

cells_dict = Dict(
    "xy" => ([-0.1 0 0.3; 0.1 0 0.3; 0.05 0.05 0.3; 0.2 0.2 0.2; -0.2 -0.2 -0.2]),
    "r" => [0.6, 0.6, 0.6, 0.6, 0.6],
    "u" => [0.8, 0.8,0.8,0.8,0.8],
    "v" => [0.8,0.8,0.8,0.8,0.8],
)

cell_ms = Cell_struct(cells_dict)
### time course data ###
names = (:xy, :r, :u, :v, :s)
data_tup = create_data_tup(names)


### simulation ###
cell_ms,data_tup = run_sim_3d(cell_ms,time_ss,par,data_tup,simulation_type_tup)
#cell_ms,data_tup = run_sim(cell_ms,time_ss,par,data_tup,simulation_type_tup)
println(length(cell_ms.r))

In [ ]:
using ColorSchemes

data = cell_ms


test = DataFrame(data.xy, :auto)
# create 4 equally spaced bins between the minimum and maximum values of the x2 column
bins = range(minimum(test.x2), stop=maximum(test.x2), length=6)

plot_list = []   
for i in 3
 
    filtered_test = filter(row ->  bins[i] <= row.x3 < bins[i+1], test)
    a = scatter(filtered_test.x1,filtered_test.x2,filtered_test.x3,label="",xlabel="",ylabel="",zlabel="",title="part "*string(i),aspect_ratio=:equal,marker_z=data.u,
    clim = (0, 1),ylim =(-4,4),xlim=(-4,4),zlim=(-4,4),colorbar = false,markersize=6)
    push!(plot_list,a)
end
    push!(plot_list,scatter(test.x1,test.x2,test.x3,label="",xlabel="",ylabel="",zlabel="",title="complete",aspect_ratio=:equal,marker_z=data.u,
    clim = (0, 1),ylim =(-4,4),xlim=(-4,4),zlim=(-4,4),colorbar = false,markersize=6))
p = plot(reverse(plot_list)...,size=(500,800),layout=(2,1),plot_title = "q="*string(par["q"])*"; Moran's I: "*string(round(calculate_moran_index(cell_ms,simulation_type_tup),digits=2)))

display(p)

In [ ]:
using ColorSchemes
u_matrix = transform_data_to_matrix(data_tup.u)
v_matrix = transform_data_to_matrix(data_tup.v)
signal_matrix = transform_data_to_matrix(data_tup.s);

### get u,v changes of cells  ###
index = []
for k in 1:length(u_matrix[1,:]) 
    input = sign.(u_matrix[:,k] - v_matrix[:,k])

    indeces = get_switches(input)
    append!(index,indeces)
end
###


# u is bigger than v -> 1

dims = string(length(cell_ms.xy[1,:]))

x = zeros(length(cell_ms.u))
x[cell_ms.u .> cell_ms.v] .= 1
x = trunc.(Int,x)

p3 = plot_time_course(x,signal_matrix,time_ss,ylabel="signal",title_1="u positve cells",title_2="v positive cells",xlabel="t [au]",size_plots=(300,500),
    size_layout=(600,500),layout_title="",label="")

p4 = plot_time_course_diff(x,u_matrix,v_matrix,time_ss,ylabel_1="u",ylabel_2="v",title_1="u positive cells",title_2="v positive cells",label_1="",label_2="",
    xlabel_2 ="t [au]",size_plots=(300,500),
    size_layout=(600,500),layout_title="")
#savefig(p_all,"plots/signal_g_0_1_q_0_"*string(par["q"])[end]*".svg")

p5 = bar(["u" "v"],[sum(x) length(x) - sum(x)],label=["u" "v"],
    title="number of cells:")
p6 = vline(time_ss.t[index],label="u,v change",title ="total u,v changes: "*string(length(index)))
plot!(p6,time_ss.t[1:length(data_tup.u)],length.(data_tup.u),color="black", label="# cells",xlabel="time",ylabel="# cells")
p7 = plot(p5,p6,layout=(2,1),size=(300,600))
plot_list= [p3,p4,p7]
p_all = plot(plot_list...,layout=(1,3),size=(1400,600),
    plot_title=dims*"d; q="*string(par["q"])*"; Moran's I: "*string(round(calculate_moran_index(cell_ms,simulation_type_tup),digits=2)))


In [ ]:
using ColorSchemes
u_matrix = transform_data_to_matrix(data_tup.u)
v_matrix = transform_data_to_matrix(data_tup.v)
signal_matrix = transform_data_to_matrix(data_tup.s);
# u is bigger than v -> 1

dims = string(length(cell_ms.xy[1,:]))

x = zeros(length(cell_ms.u))
x[cell_ms.u .> cell_ms.v] .= 1
x = trunc.(Int,x)

p3 = plot_time_course(x,signal_matrix,time_ss,ylabel="signal",title_1="u positve cells",title_2="v positive cells",xlabel="t [au]",size_plots=(300,500),
    size_layout=(600,500),layout_title="",label="")

p4 = plot_time_course_diff(x,u_matrix,v_matrix,time_ss,ylabel_1="u",ylabel_2="v",title_1="u positive cells",title_2="v positive cells",label_1="",label_2="",
    color_1=ColorSchemes.bam10[1],color_2=ColorSchemes.bam10[9],xlabel_2 ="t [au]",size_plots=(300,500),
    size_layout=(600,500),layout_title="")
#savefig(p_all,"plots/signal_g_0_1_q_0_"*string(par["q"])[end]*".svg")

p5 = bar(["u" "v"],[sum(x) length(x) - sum(x)],color=[ColorSchemes.bam10[1] ColorSchemes.bam10[9]],label=["u" "v"],
    title="number of cells:")
p6 = plot_2d(cell_ms,colorbar=false)
p7 = plot(p5,p6,layout=(2,1),size=(300,600),margin=(5,:mm))
plot_list= [p3,p4,p7]
p_all = plot(plot_list...,layout=(1,3),size=(1400,600),margin=(5,:mm),
    plot_title=dims*"d; q="*string(par["q"])*"; Moran's I: "*string(round(calculate_moran_index(cell_ms,simulation_type_tup),digits=2)))
#savefig(p_all,"plots/gamma_1.svg")

In [ ]:
u_sum = [sum([x for x in u_matrix[y,:] if !ismissing(x)]) for y in range(1,length(u_matrix[:,1]))]
signal_sum = [sum([x for x in signal_matrix[y,:] if !ismissing(x)]) for y in range(1,length(signal_matrix[:,1]))]
a = plot(range(1,length(signal_matrix[:,1])), u_sum - signal_sum, xlabel="timesteps",label="u - signal",title="u -signal per timestep;no topographie")
#savefig(a,"timestep_no_topography.svg")

In [ ]:

function calculate_infl_matrix_2(adj_matrix,par) # distance based signalling

    # Packages: LinearAlgebra
    
   distance = zeros(dim(adj_matrix),dim(adj_matrix))
   for i in range(1,dim(adj_matrix))
       distance[i,:]= dijkstra_from_matrix(adj_matrix,i)
   end

   distance = remove_diagonal(distance)
   infl_matrix = zeros(length(distance[:,1]),length(distance[1,:]))
   
   for  i in range(1,length(distance[:,1]))
       infl_matrix[i,:] = par["q"] .^ (distance[i,:].-1) * (maximum(sum(par["q"] .^(distance[i,:].-1))))^(-1)
   end
    """
    scaling = (maximum(sum(par["q"] .^(distance.-1),dims=2)))^(-1)
    for  i in range(1,length(distance[:,1]))
        infl_matrix[i,:] = par["q"] .^ (distance[i,:].-1) * scaling
    end
    """
   infl_matrix = round.(add_diagonal(infl_matrix),digits = 4)

   return infl_matrix
    
end

In [ ]:
using GraphRecipes


par["q"] = 0.1
infl_matrix = calculate_infl_matrix(matrix,par,simulation_type_tup)
scaling = []

for i in range(1,length(infl_matrix[1,:]))
   # println(round(sum(infl_matrix[:,i]),digits=2))
    push!(scaling,round(sum(infl_matrix[i,:]),digits=2))
end
index_max = findall(a-> a == maximum(scaling),scaling)[1]
names = round.(infl_matrix[1,:],digits=2)

colors = ["seagreen","green2","lightgreen"]
color_dict = Dict(zip(sort(collect(Set(names))),colors))
color_names = []
for i in names
    push!(color_names,color_dict[i])
end


g1 = graphplot(matrix,curves=false,names=scaling,markersize=0.2,markercolor="seagreen")

g2 = graphplot(matrix,curves=false,names=names,markersize=0.2,markercolor=color_names)

names = round.(infl_matrix[index_max,:],digits=2)

colors = ["seagreen","green2","lightgreen"]
color_dict = Dict(zip(sort(collect(Set(names))),colors))
color_names = []
for i in names
    push!(color_names,color_dict[i])
end
g3 = graphplot(matrix,curves=false,names=names,markersize=0.2,markercolor=color_names)

par["q"] = 0.1
infl_matrix_2 = calculate_infl_matrix_2(matrix,par)
scaling_2 = []

for i in range(1,length(infl_matrix[1,:]))
   # println(round(sum(infl_matrix[:,i]),digits=2))
    push!(scaling_2,round(sum(infl_matrix_2[i,:]),digits=2))
end

names_2 = round.(infl_matrix_2[1,:],digits=2)

colors = ["dodgerblue","lightblue3","lightblue2"]
color_dict_2 = Dict(zip(sort(collect(Set(names_2))),colors))
color_names_2 = []
for i in names_2
    push!(color_names_2,color_dict_2[i])
end

g4 = graphplot(matrix,curves=false,names=scaling_2,markersize=0.2,markercolor="dodgerblue")
g5 = graphplot(matrix,curves=false,names= names_2,markersize=0.2,markercolor=color_names_2);

names_2 = round.(infl_matrix_2[index_max,:],digits=2)

colors = ["dodgerblue","lightblue3","lightblue2"]
color_dict_2 = Dict(zip(sort(collect(Set(names_2))),colors))
color_names_2 = []
for i in names_2
    push!(color_names_2,color_dict_2[i])
end
g6 = graphplot(matrix,curves=false,names= names_2,markersize=0.2,markercolor=color_names_2);
p_all = plot(g1,g2,g3,g4,g5,g6,layout=(2,3),size=(1200,800),plot_title="Green = Simon's scaling - blue Sascha's")
#savefig(p_all,"scaling_difference.svg")

In [ ]:
findall(a-> a == maximum(scaling),scaling)

In [ ]:
cell_number = 3
u_matrix = transform_data_to_matrix(data_tup.u)
v_matrix = transform_data_to_matrix(data_tup.v)
plot(time_ss.t[1:length(u_matrix[:,1])],u_matrix[:,cell_number],label="u",title="expression level of cell " * string(cell_number),xlabel="t [au]",ylabel="[au]")
plot!(time_ss.t[1:length(u_matrix[:,1])],v_matrix[:,cell_number],label="v")

In [ ]:



simulation_type_tup = (

    # simulation steps:

    physics = true,
    brownian = true,
    transcr = true,
    cell_division = true,
    affinity = true,
    anim = false,
    save_data = true,
    track_progress = false,

    # specification of which functions to use:

    # physics: 
        # cell devision, can be random_angle_cell_devision,  non_random_cell_devision
    cell_div_func = random_angle_cell_devision,
    func_orth = false,
    
    # transcription:
        #signalling: can be nearest_neighbour, distance_based
    sign_func = signalling_distance,
    check_distance = true,

    # animation:
        #scatter plot is default
    draw_edges = false,
    draw_radius = false,

    # simulate to specific number of cells?
    cell_number_check = true

)
### Paramater ###
par = CSV.File("standard_parameter.csv") |> Dict
### Time ###
time_ss = Time_struct(30,100)
### Initial conditions 2d ###
"""
cell_ms = deserialize("3d_initial_50_cells.jld2")
cell_ms.r0 = cell_ms.r
cell_ms.t0 = zeros(length(cell_ms.r))

### randomize u and v ###
#cell_ms.u = 0.1*rand(length(cell_ms.r))
#cell_ms.v = 0.1*rand(length(cell_ms.r))

cell_ms.v = fill(0.1,length(cell_ms.r))
cell_ms.u = fill(0.1,length(cell_ms.r))
"""
###  Inititial conditions 3d ###

cells_dict = Dict(
    "xy" => ([-0.1 0 0.3; 0.1 0 0.3; 0.05 0.05 0.3; 0.2 0.2 0.2; -0.2 -0.2 -0.2]),
    "r" => [0.6, 0.6, 0.6, 0.6, 0.6],
    "u" => [0.1, 0.1,0.1, 0.1, 0.1],
    "v" => [0.1, 0.1,0.1, 0.1, 0.1],
)
cell_ms = Cell_struct(cells_dict)

par["q"] = 0.9
### time course data ###
names = (:xy, :r, :u, :v, :s)
data_tup_3d = create_data_tup(names)


### simulation ###
cell_ms,data_tup_3d = run_sim_3d(cell_ms,time_ss,par,data_tup_3d,simulation_type_tup)
println(length(cell_ms.r))

In [ ]:
u_matrix = transform_data_to_matrix(data_tup_3d.u)
v_matrix = transform_data_to_matrix(data_tup_3d.v)
signal_matrix = transform_data_to_matrix(data_tup_3d.s);
# u is bigger than v -> 1
x = zeros(length(cell_ms.u))
x[cell_ms.u .> cell_ms.v] .= 1
x = trunc.(Int,x)
p3 = plot_time_course(x,signal_matrix,ylabel="signal",title_1="u positve cells",title_2="v positive cells",xlabel="t [au]",size_plots=(300,500),size_layout=(600,500),
layout_title="signal each cells receives; start= 5 cells; q="*string(par["q"]),label="")

p4 = plot_time_course_diff(x,u_matrix,v_matrix,ylabel_1="u",ylabel_2="v",title_1="u positive cells",title_2="v positive cells",label_1="",label_2="",
color_1=ColorSchemes.bam10[1],color_2=ColorSchemes.bam10[9],xlabel="t [au]",size_plots=(300,500),size_layout=(600,500),layout_title="u and v expression; start= 5 cells; q="*string(par["q"]))
plot_list= [p3,p4]
p_all = plot(p3,p4,layout=(1,2),size=(1000,600),left_margin=(5,:mm))

In [ ]:
p3 = plot_time_course_diff(x,signal_matrix,u_matrix,ylabel_1="signal",ylabel_2="u",title_1="",title_2="",label_1="",label_2="",values = [1,1],
color_1="black",color_2=ColorSchemes.bam10[1],xlabel_1="t [au]",xlabel_2="t [au]",size_plots=(400,300),size_layout=(1000,300),layout_title="u positive cells",layout=(1,2),margin=(5,:mm))

p4 = plot_time_course_diff(x,signal_matrix,v_matrix,ylabel_1="signal",ylabel_2="v",title_1="",title_2="",label_1="",label_2="",values=[0,0],
color_1="black",color_2=ColorSchemes.bam10[9],xlabel_1="t [au]",xlabel_2="t [au]",size_plots=(400,300),size_layout=(1000,300),layout_title="v positive cells",layout=(1,2),margin=(5,:mm))

p_all = plot(p3,p4,layout=(2,1),size=(1000,600))
savefig(p_all,"test2.svg")


In [ ]:
### plot u over time ###

u_matrix = transform_data_to_matrix(data_tup_3d.u)

# u is bigger than v -> 1
x[cell_ms.u .> cell_ms.v] .= 1
x = trunc.(Int,x)

p = plot(ylabel="u level",title="u positve cells",ylim=(0,0.1))
p2 = plot(ylabel="u level",xlabel="t [au]",title="v positive cells",ylim=(0,0.1))
for i in range(1,length(cell_ms.r),step=1)
    if x[i] == 1 
        plot!(p,time_ss.t[1:length(signal_matrix[:,1])],u_matrix[:,i],label="",color="red",alpha=0.5)
    else
        plot!(p2,time_ss.t[1:length(signal_matrix[:,1])],u_matrix[:,i],label="",color="blue",alpha=0.5)
    end
end
p4 = plot(p,p2,layout=(2,1),size=(600,500),plot_title="u level in each cell; start= 5 cells; q="*string(par["q"]),plot_titlefontsize=10)


In [ ]:
### plot v over time ###

u_matrix = transform_data_to_matrix(data_tup_3d.u)
v_matrix = transform_data_to_matrix(data_tup_3d.v)

# u is bigger than v -> 1
x = zeros(length(cell_ms.u))
x[cell_ms.u .> cell_ms.v] .= 1
x = trunc.(Int,x)

p = plot(ylabel="v level",title="u positve cells",ylim=(0,0.1))
p2 = plot(ylabel="v level",xlabel="t [au]",title="v positive cells",ylim=(0,0.1))
for i in range(1,length(cell_ms.r),step=1)
    if x[i] == 1 
        plot!(p,time_ss.t[1:length(signal_matrix[:,1])],v_matrix[:,i],label="",color="red",alpha=0.5)
    else
        plot!(p2,time_ss.t[1:length(signal_matrix[:,1])],v_matrix[:,i],label="",color="blue",alpha=0.5)
    end
end
p5 = plot(p,p2,layout=(2,1),size=(600,500),plot_title=" v level in each cell; start= 5 cells; q="*string(par["q"]),plot_titlefontsize=10)

In [ ]:
plot_list= [p3,p4,p5]
#display(p3)
#display(p4)
#display(p5)
p_all = plot(plot_list...,layout=(1,3),size=(1600,500),left_margin=(8,:mm),bottom_margin=(8,:mm));
#savefig(p_all,"signal_u_v_3d_q_0_"*string(par["q"])[3]*".svg")

In [ ]:
u_matrix = transform_data_to_matrix(data_tup_3d.u)
v_matrix = transform_data_to_matrix(data_tup_3d.v)

# u is bigger than v -> 1
x[cell_ms.u .> cell_ms.v] .= 1
x = trunc.(Int,x)

p21 = plot(ylabel="expression level",xlabel="t [au]",title="expression and signal level q=0."*string(par["q"]),ylim=(0,0.1))
#p22 = plot(ylabel="level",xlabel="t [au]",title="expression and signal level v positive",ylim=(0,0.1))
u_marker = 0
v_marker = 0
for i in range(1,5,step=1)
    if x[i] == 1 
        if u_marker == 0

            plot!(time_ss.t[1:501],u_matrix[1:501,i],label="u level",color="red",linewidth=4,style=:dot)
            plot!(time_ss.t[1:501],signal_matrix[1:501,i],label="u signal",color="darkred",linewidth=4,style=:dot)

            u_marker = 1
        else
        #plot!(time_ss.t[1:501],v_matrix[1:501,i],label="",color="red",linewidth=4)
            plot!(time_ss.t[1:501],u_matrix[1:501,i],label="",color="red",linewidth=4,style=:dot)
            plot!(time_ss.t[1:501],signal_matrix[1:501,i],label="",color="darkred",linewidth=4,style=:dot)
        end
    else

        if v_marker == 0
            plot!(time_ss.t[1:501],v_matrix[1:501,i],label="v level",color="blue",linewidth=4,style=:dashdot)
            plot!(time_ss.t[1:501],signal_matrix[1:501,i],label="v signal",color="darkblue",linewidth=4,style=:dashdot)
            v_marker = 1
        else
            plot!(time_ss.t[1:501],v_matrix[1:501,i],label="",color="blue",linewidth=4,style=:dashdot)
            plot!(time_ss.t[1:501],signal_matrix[1:501,i],label="",color="darkblue",linewidth=4,style=:dashdot)
        end

    end
end
p21
#savefig(p21,"first_5t_q=0."*string(par["q"])[3]*".svg")


In [ ]:
time_ss.t[501]

In [ ]:
simulation_type_tup = (

    # simulation steps:

    physics = true,
    brownian = true,
    transcr = true,
    cell_division = true,
    affinity = true,
    anim = true,
    save_data = true,
    track_progress = false,

    # specification of which functions to use:

    # physics: 
        # cell devision, can be random_angle_cell_devision,  non_random_cell_devision
    cell_div_func = random_angle_cell_devision,
    func_orth = false,
    
    # transcription:
        #signalling: can be nearest_neighbour, distance_based
    sign_func = signalling_distance,
    check_distance = false,

    # animation:
        #scatter plot is default
    draw_edges = false,
    draw_radius = false,

    # simulate to specific number of cells?
    cell_number_check = true

)

degree_list = []
for i in range(1,length(signal_matrix[:,1]))
    data_tup_3d.xy[i]
    connectivity = delaunay(data_tup_3d.xy[i]')
    g = matrix_from_delaunay(connectivity,data_tup_3d,simulation_type_tup)
    sum_list = []
    for j in range(1,length(g[:,1]))
        push!(sum_list,sum(g[:,j]) )
    end
    push!(degree_list,sum_list)
end
matrix =transform_data_to_matrix(degree_list);

In [ ]:
p = plot(ylabel="neighbours",title="u positve cells",ylim=(0,20))
p2 = plot(ylabel="neighbours",xlabel="t [au]",title="v positive cells",ylim=(0,20))
for i in range(1,length(cell_ms.r),step=1)
    if x[i] == 1 
        plot!(p,time_ss.t[1:length(signal_matrix[:,1])],matrix[:,i],label="",color="red",alpha=0.5)
    else
        plot!(p2,time_ss.t[1:length(signal_matrix[:,1])],matrix[:,i],label="",color="blue",alpha=0.5)
    end
end
p3 = plot(p,p2,layout=(2,1),size=(600,500),plot_title="# of neighbours; start= 5 cells",
    titlefontsize=10,legendfontsize=10,guidefontsize=10,legend=:bottomright)